# Data Description
- Background: 民政局依戶籍登記資料產製的人口統計；以行政里人口結構支援 Dataset B1。里界僅為本次記憶體名稱比對的輔助輸入，其正式文件另見里界 Notebook。
- date received: 未知；本機下載／收到日期待學生或原始下載紀錄補證，不以文件建立日代填。
- Path to data file: [臺北市各里人口數按年齡分官方資料集](https://data.taipei/dataset/detail?id=a6394e3f-3514-4542-87bd-de4310a40db3)；來源網址由學生提供，2026-11-25 唯讀查閱詮釋資料，未重新下載原始資料。PDF 下載來源與本機位元組關係未認證。
- Unit of observation: 年×月×區域×性別；同時含市、行政區、行政里，不可直接跨層級加總。
- Sample period: 本機民國 112 年 1 月至 115 年 10 月；研究關注 113、114 年共 24 個月。
- Known issues: 人口里代碼 11 碼而 CPTVID 為 10 碼；糖?里／糖蔀里例外仍須正式來源與 crosswalk。設籍人口不是事故交通暴露人口。
- Definition for each variable: 見 Summary Stats 的完整逐欄定義表；「觀測內容」與「官方定義」分列，官方未定義處明記未知。
- Data path: [112迄今各里人口數按年齡分.csv](112迄今各里人口數按年齡分.csv)、[G97_A_CAVLGE_P.shp](G97_A_CAVLGE_P.shp)、[G97_A_CAVLGE_P.shx](G97_A_CAVLGE_P.shx)、[G97_A_CAVLGE_P.dbf](G97_A_CAVLGE_P.dbf)、[G97_A_CAVLGE_P.prj](G97_A_CAVLGE_P.prj)、[G97_A_CAVLGE_P.cpg](G97_A_CAVLGE_P.cpg)；程式讀檔使用下方精確的工作包相對路徑。
- Role: `raw`（參考 PDF 亦逐檔登記）
- Verification status: 已實際執行唯讀文件化檢查；詳見保存輸出。檔案身分、基本結構及列明診斷可核對，不認證為 analysis-ready。
- SHA-256: 見下方固定綁定值及 Summary 逐檔大小／hash 表；每次讀檔前後重新計算。
- Next processing notebook: pending；尚未建立，不預設處理檔名。
- Author / contributors: 詹明翰、謝子詮、曾翊涵（學生確認三人共同完成本次 Unit 0.5；不推定個別分工）
- Created At: 2026-11-25（文件建立日，與 date received 不同）

**官方版本資訊與本機版本分開記錄：** 官方提供機關為民政局，資源更新 2026-11-03 10:49:10、詮釋資料更新 2026-11-03 10:49:22。這是查閱當時網頁時間；本機下載日期及是否同版本未核對。官方頁僅列欄名與戶籍統計說明，未取得本機月基準日及完整代碼規格。

本文件依 `templates/data/raw/template-doc-and-summary-stats.ipynb` 的原欄位與章節順序建立。
僅描述未修改原始資料；未刪除、補值、修復、正式轉換或輸出 processed。正式研究與規劃定義詳見 [PROJECT.md](../../PROJECT.md)。


In [1]:
from pathlib import Path
import hashlib, json, sys, platform, os
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / 'AGENTS.md').is_file() and (candidate / 'templates').is_dir():
            return candidate
    raise FileNotFoundError('Open within the project containing AGENTS.md and templates/.')

PROJECT_ROOT = find_project_root()
assert Path(sys.executable).resolve() == (PROJECT_ROOT / '.venv/Scripts/python.exe').resolve()
assert sys.prefix != sys.base_prefix
TEMP = PROJECT_ROOT / 'data/temp/unit05_documentation'
for variable, folder in {'MPLCONFIGDIR':'matplotlib', 'IPYTHONDIR':'ipython',
                         'JUPYTER_RUNTIME_DIR':'runtime'}.items():
    location = TEMP / folder
    location.mkdir(parents=True, exist_ok=True)
    os.environ[variable] = str(location)
os.environ['PROJ_NETWORK'] = 'OFF'
pd.set_option('display.max_colwidth', 100)

def project_file(relative_path, area='raw'):
    rel = Path(relative_path)
    if rel.is_absolute() or '..' in rel.parts:
        raise ValueError('Explicit project-relative path required')
    path = (PROJECT_ROOT / rel).resolve()
    if not path.is_relative_to((PROJECT_ROOT / 'data' / area).resolve()):
        raise ValueError('Input path outside permitted data area')
    if not path.is_file():
        raise FileNotFoundError(path)
    return path

def sha256_file(path):
    h = hashlib.sha256()
    with path.open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

def verify_inputs(expected):
    rows = []
    for relative_path, digest in expected.items():
        path = project_file(relative_path)
        actual = sha256_file(path)
        if actual != digest:
            raise ValueError(f'Input changed; review provenance before loading: {relative_path}')
        rows.append({'Data path':relative_path, 'bytes':path.stat().st_size,
                     'format':path.suffix.lower(), 'SHA-256':actual})
    return pd.DataFrame(rows)

def show_table(title, frame, block=25):
    display(Markdown('**' + title + '**'))
    if not len(frame):
        display(Markdown('無符合紀錄。'))
    else:
        for start in range(0, len(frame), block):
            display(frame.iloc[start:start + block])

def sample_ten(title, frame, block_columns=12):
    display(Markdown('**' + title + '：實際前十列，按欄分組呈現；未抽樣或排序**'))
    for start in range(0, len(frame.columns), block_columns):
        display(frame.iloc[:10, start:start + block_columns])

def schema(frame, definitions, levels=None):
    rows=[]
    for column in frame.columns:
        meaning, evidence = definitions[column]
        rows.append({'欄位':column, '讀入型態':str(frame[column].dtype),
                     '定義或可觀察內容':meaning, '定義依據及限制':evidence,
                     '層級':levels.get(column, '原始紀錄') if levels else '原始紀錄',
                     '缺失數':int(frame[column].isna().sum()),
                     '非缺失相異值':int(frame[column].nunique(dropna=True))})
    return pd.DataFrame(rows)

def finish_check():
    for relative_path,digest in EXPECTED_SHA256.items():
        assert sha256_file(project_file(relative_path)) == digest, relative_path
    print({'input_hashes_unchanged':True, 'input_files_checked':len(EXPECTED_SHA256),
           'interpreter_matches_project_venv':True, 'python_version':platform.python_version(),
           'scope':'read-only raw documentation; no processed dataset or model output'})


## Change for Your Own Data
以下固定原始檔路徑與事前檢查所得 SHA-256；版本不同即停止，不換用其他檔案。
CSV 的編碼／代碼型態、Shapefile 的 reader 與配套檔均明列。讀入型態是本次 reader 的推斷或明示設定，不冒稱官方儲存型態。


In [2]:
EXPECTED_SHA256 = {'data/raw/112迄今各里人口數按年齡分.csv': 'a663a4f0329b18bb0514da7fd9f6845312d08fadef71e566faf1930c03541641', 'data/raw/G97_A_CAVLGE_P.shp': 'f7fb4813a2283198ca564f5686823c454c5fd832d2f6981a2ee88a94719be3cc', 'data/raw/G97_A_CAVLGE_P.shx': 'ab7bc5e63229f676d3580b68e98db59b91a2d327c74ca23b0000ca1c568d414e', 'data/raw/G97_A_CAVLGE_P.dbf': '2f43f92e84279a45b2512cf453e9b72f706d1ad1402665c946b1a354f75d8577', 'data/raw/G97_A_CAVLGE_P.prj': '8dac7d82ea889063f9eaa7df408fcaeb7307d32831ea973b6534d0139a4523c5', 'data/raw/G97_A_CAVLGE_P.cpg': '3ad3031f5503a4404af825262ee8232cc04d4ea6683d42c5dd0a2f2a27ac9824'}
INPUT_MANIFEST = verify_inputs(EXPECTED_SHA256)


## Summary


In [3]:
show_table('表 1：原始輸入逐檔檔案格式、大小及 SHA-256', INPUT_MANIFEST)
df=pd.read_csv(project_file('data/raw/112迄今各里人口數按年齡分.csv'),encoding='cp950',dtype={'區域代碼':'string'},low_memory=False)
print({'rows':len(df),'columns':len(df.columns),'encoding':'CP950','region_code_reader_dtype':'string; preserve original code text'})


**表 1：原始輸入逐檔檔案格式、大小及 SHA-256**

,Data path,bytes,format,SHA-256
0,data/raw/112迄今各里人口數按年齡分.csv,21284860,.csv,a663a4f0329b18bb0514da7fd9f6845312d08fadef71e566faf1930c03541641
1,data/raw/G97_A_CAVLGE_P.shp,765784,.shp,f7fb4813a2283198ca564f5686823c454c5fd832d2f6981a2ee88a94719be3cc
2,data/raw/G97_A_CAVLGE_P.shx,3748,.shx,ab7bc5e63229f676d3580b68e98db59b91a2d327c74ca23b0000ca1c568d414e
3,data/raw/G97_A_CAVLGE_P.dbf,272706,.dbf,2f43f92e84279a45b2512cf453e9b72f706d1ad1402665c946b1a354f75d8577
4,data/raw/G97_A_CAVLGE_P.prj,399,.prj,8dac7d82ea889063f9eaa7df408fcaeb7307d32831ea973b6534d0139a4523c5
5,data/raw/G97_A_CAVLGE_P.cpg,5,.cpg,3ad3031f5503a4404af825262ee8232cc04d4ea6683d42c5dd0a2f2a27ac9824


{'rows': 64722, 'columns': 107, 'encoding': 'CP950', 'region_code_reader_dtype': 'string; preserve original code text'}


### Sample


In [4]:
sample_ten('表 3：人口原始樣本',df)


**表 3：人口原始樣本：實際前十列，按欄分組呈現；未抽樣或排序**

,年份,月份,區域代碼,區域別,性別,總計,0歲數量,1歲數量,2歲數量,3歲數量,4歲數量,5歲數量
0,112,1,63000000,總計,計,2488043,13244,16616,17363,19406,20452,21942
1,112,1,63000000,總計,男,1180304,6784,8539,9012,9961,10590,11291
2,112,1,63000000,總計,女,1307739,6460,8077,8351,9445,9862,10651
3,112,1,63000010,松山區,計,190772,938,1267,1281,1463,1589,1708
4,112,1,63000010,松山區,男,89059,482,654,689,767,829,863
5,112,1,63000010,松山區,女,101713,456,613,592,696,760,845
6,112,1,63000010002,莊敬里,計,5056,29,36,37,42,39,34
7,112,1,63000010002,莊敬里,男,2441,15,19,22,22,21,16
8,112,1,63000010002,莊敬里,女,2615,14,17,15,20,18,18
9,112,1,63000010003,東榮里,計,7846,27,42,42,54,48,71


,6歲數量,7歲數量,8歲數量,9歲數量,10歲數量,11歲數量,12歲數量,13歲數量,14歲數量,15歲數量,16歲數量,17歲數量
0,23100,23413,24143,21780,26507,23360,19553,21106,20410,21130,19918,19411
1,11954,12214,12327,11249,13680,12189,10191,10891,10654,10952,10282,10300
2,11146,11199,11816,10531,12827,11171,9362,10215,9756,10178,9636,9111
3,1892,1963,2152,2091,2652,2372,1996,1904,1773,1728,1545,1471
4,959,1028,1054,1096,1370,1206,1071,951,930,928,767,792
5,933,935,1098,995,1282,1166,925,953,843,800,778,679
6,40,50,45,41,47,34,30,32,33,30,47,27
7,19,24,18,20,31,14,18,15,17,17,23,13
8,21,26,27,21,16,20,12,17,16,13,24,14
9,79,94,103,129,142,159,114,112,87,62,66,54


,18歲數量,19歲數量,20歲數量,21歲數量,22歲數量,23歲數量,24歲數量,25歲數量,26歲數量,27歲數量,28歲數量,29歲數量
0,19663,19751,21512,21734,26333,25025,23332,27827,27887,29287,28633,28857
1,10315,10216,11205,11191,13675,12971,12117,14271,14180,14887,14505,14390
2,9348,9535,10307,10543,12658,12054,11215,13556,13707,14400,14128,14467
3,1473,1440,1609,1582,1983,1852,1637,1983,1990,2096,1978,1973
4,772,773,824,828,1025,953,857,977,1028,1086,984,1025
5,701,667,785,754,958,899,780,1006,962,1010,994,948
6,49,43,56,37,64,43,48,54,53,55,77,63
7,25,27,31,18,29,22,18,27,21,32,36,32
8,24,16,25,19,35,21,30,27,32,23,41,31
9,59,61,56,54,88,71,71,72,84,87,73,86


,30歲數量,31歲數量,32歲數量,33歲數量,34歲數量,35歲數量,36歲數量,37歲數量,38歲數量,39歲數量,40歲數量,41歲數量
0,28875,28948,31518,29675,33970,30847,30481,35032,38312,39625,42076,43693
1,14437,14482,15573,14450,16174,14674,14071,16434,18021,18584,19931,20403
2,14438,14466,15945,15225,17796,16173,16410,18598,20291,21041,22145,23290
3,2042,1947,2186,2069,2326,2238,2042,2450,2797,2875,3095,3223
4,994,945,1072,960,1076,1049,933,1092,1261,1315,1432,1414
5,1048,1002,1114,1109,1250,1189,1109,1358,1536,1560,1663,1809
6,49,49,58,61,75,58,47,66,67,84,95,102
7,22,26,29,27,36,32,21,32,32,44,43,52
8,27,23,29,34,39,26,26,34,35,40,52,50
9,73,87,70,76,68,94,64,80,99,104,115,142


,42歲數量,43歲數量,44歲數量,45歲數量,46歲數量,47歲數量,48歲數量,49歲數量,50歲數量,51歲數量,52歲數量,53歲數量
0,42738,43522,42222,38174,45029,37098,36758,36214,35852,36910,38092,38046
1,20032,20197,19713,17704,21008,16955,16971,17047,16568,17017,17564,17363
2,22706,23325,22509,20470,24021,20143,19787,19167,19284,19893,20528,20683
3,3288,3349,3178,2967,3601,2920,2924,2819,2789,2911,2852,2744
4,1448,1545,1446,1312,1601,1258,1322,1224,1284,1355,1281,1253
5,1840,1804,1732,1655,2000,1662,1602,1595,1505,1556,1571,1491
6,98,103,68,74,96,80,70,86,71,95,67,89
7,45,55,27,36,47,38,30,40,30,50,32,42
8,53,48,41,38,49,42,40,46,41,45,35,47
9,133,143,138,133,169,116,119,100,97,102,98,102


,54歲數量,55歲數量,56歲數量,57歲數量,58歲數量,59歲數量,60歲數量,61歲數量,62歲數量,63歲數量,64歲數量,65歲數量
0,37441,35963,36804,37231,38051,38269,37478,36711,37407,36879,37589,35655
1,17136,16564,16946,17240,17513,17540,17213,16868,17199,16733,17214,15994
2,20305,19399,19858,19991,20538,20729,20265,19843,20208,20146,20375,19661
3,2739,2712,2670,2666,2734,2769,2685,2693,2660,2612,2731,2573
4,1141,1271,1210,1140,1258,1212,1162,1246,1212,1145,1187,1120
5,1598,1441,1460,1526,1476,1557,1523,1447,1448,1467,1544,1453
6,65,70,79,73,70,87,66,72,61,71,73,73
7,33,36,32,37,38,40,37,40,39,38,38,27
8,32,34,47,36,32,47,29,32,22,33,35,46
9,110,111,106,95,99,108,110,100,101,111,102,92


,66歲數量,67歲數量,68歲數量,69歲數量,70歲數量,71歲數量,72歲數量,73歲數量,74歲數量,75歲數量,76歲數量,77歲數量
0,37112,36605,34813,33588,33428,32993,29096,27411,23299,20804,16197,13033
1,16913,16947,16090,15367,15273,14930,12994,12233,10504,9238,7360,5633
2,20199,19658,18723,18221,18155,18063,16102,15178,12795,11566,8837,7400
3,2745,2752,2585,2606,2562,2628,2382,2248,1977,1832,1498,1133
4,1221,1286,1181,1160,1124,1148,1106,984,851,824,669,482
5,1524,1466,1404,1446,1438,1480,1276,1264,1126,1008,829,651
6,84,84,66,75,62,51,81,64,46,60,43,38
7,49,43,33,37,21,26,28,32,14,28,17,14
8,35,41,33,38,41,25,53,32,32,32,26,24
9,104,109,115,123,127,104,105,104,83,77,68,43


,78歲數量,79歲數量,80歲數量,81歲數量,82歲數量,83歲數量,84歲數量,85歲數量,86歲數量,87歲數量,88歲數量,89歲數量
0,14812,14584,13707,12964,12334,10922,9757,8552,7529,6870,5900,5295
1,6409,6223,5834,5256,5006,4265,3786,3200,2790,2500,2293,2063
2,8403,8361,7873,7708,7328,6657,5971,5352,4739,4370,3607,3232
3,1291,1259,1128,1095,1079,834,798,693,605,582,496,456
4,574,564,514,476,467,350,305,242,226,209,184,197
5,717,695,614,619,612,484,493,451,379,373,312,259
6,44,44,28,25,27,19,20,15,13,10,16,9
7,17,19,10,11,16,10,6,5,3,5,5,4
8,27,25,18,14,11,9,14,10,10,5,11,5
9,66,63,54,67,51,27,40,35,25,21,15,23


,90歲數量,91歲數量,92歲數量,93歲數量,94歲數量,95歲數量,96歲數量,97歲數量,98歲數量,99歲數量,100歲以上
0,4710,4239,3564,2999,2363,1888,1460,1016,735,512,1072
1,1910,1817,1556,1323,1047,851,638,478,319,211,531
2,2800,2422,2008,1676,1316,1037,822,538,416,301,541
3,395,397,292,267,202,213,138,94,71,55,127
4,179,160,123,118,74,95,68,46,32,24,52
5,216,237,169,149,128,118,70,48,39,31,75
6,13,9,8,8,4,2,1,2,2,0,0
7,7,2,5,4,3,1,0,1,0,0,0
8,6,7,3,4,1,1,1,1,2,0,0
9,17,13,8,10,9,9,5,5,2,1,4


### Summary Stats


In [5]:
DEFINITIONS = {'年份': ('原始年碼為民國年，112–115；研究使用 113、114', '官方頁列年份；實際值及年月完整性見輸出'), '月份': ('人口統計月份；月內基準日／結算方式尚未取得', '官方頁列月份，未提供本機版本的詳細基準說明'), '區域代碼': ('字串保留原碼；本機市／區為 8 碼、里為 11 碼', '本次依原碼長度、區域別及前綴的檔內診斷；正式代碼規格仍待官方文件'), '區域別': ('區域名稱；市總計、行政區或行政里', '原始值觀測；糖?里為原字串，不以換編碼修補'), '性別': ('計、男、女；計為合計，不能再次與男／女相加', '原始分類及逐欄男＋女＝計檢查'), '總計': ('該年／月／區域／性別的設籍人口合計', '官方說明依戶籍登記產製；月內基準與本機版本尚待確認'), '0歲數量': ('欄位標示為 0 歲人口數；單位人', '官方頁列逐歲欄名；精確年齡計算及月基準定義未取得'), '1歲數量': ('欄位標示為 1 歲人口數；單位人', '官方頁列逐歲欄名；精確年齡計算及月基準定義未取得'), '2歲數量': ('欄位標示為 2 歲人口數；單位人', '官方頁列逐歲欄名；精確年齡計算及月基準定義未取得'), '3歲數量': ('欄位標示為 3 歲人口數；單位人', '官方頁列逐歲欄名；精確年齡計算及月基準定義未取得'), '4歲數量': ('欄位標示為 4 歲人口數；單位人', '官方頁列逐歲欄名；精確年齡計算及月基準定義未取得'), '5歲數量': ('欄位標示為 5 歲人口數；單位人', '官方頁列逐歲欄名；精確年齡計算及月基準定義未取得'), '6歲數量': ('欄位標示為 6 歲人口數；單位人', '官方頁列逐歲欄名；精確年齡計算及月基準定義未取得'), '7歲數量': ('欄位標示為 7 歲人口數；單位人', '官方頁列逐歲欄名；精確年齡計算及月基準定義未取得'), '8歲數量': ('欄位標示為 8 歲人口數；單位人', '官方頁列逐歲欄名；精確年齡計算及月基準定義未取得'), '9歲數量': ('欄位標示為 9 歲人口數；單位人', '官方頁列逐歲欄名；精確年齡計算及月基準定義未取得'), '10歲數量': ('欄位標示為 10 歲人口數；單位人', '官方頁列逐歲欄名；精確年齡計算及月基準定義未取得'), '11歲數量': ('欄位標示為 11 歲人口數；單位人', '官方頁列逐歲欄名；精確年齡計算及月基準定義未取得'), '12歲數量': ('欄位標示為 12 歲人口數；單位人', '官方頁列逐歲欄名；精確年齡計算及月基準定義未取得'), '13歲數量': ('欄位標示為 13 歲人口數；單位人', '官方頁列逐歲欄名；精確年齡計算及月基準定義未取得'), '14歲數量': ('欄位標示為 14 歲人口數；單位人', '官方頁列逐歲欄名；精確年齡計算及月基準定義未取得'), '15歲數量': ('欄位標示為 15 歲人口數；單位人', '官方頁列逐歲欄名；精確年齡計算及月基準定義未取得'), '16歲數量': ('欄位標示為 16 歲人口數；單位人', '官方頁列逐歲欄名；精確年齡計算及月基準定義未取得'), '17歲數量': ('欄位標示為 17 歲人口數；單位人', '官方頁列逐歲欄名；精確年齡計算及月基準定義未取得'), '18歲數量': ('欄位標示為 18 歲人口數；單位人', '官方頁列逐歲欄名；精確年齡計算及月基準定義未取得'), '19歲數量': ('欄位標示為 19 歲人口數；單位人', '官方頁列逐歲欄名；精確年齡計算及月基準定義未取得'), '20歲數量': ('欄位標示為 20 歲人口數；單位人', '官方頁列逐歲欄名；精確年齡計算及月基準定義未取得'), '21歲數量': ('欄位標示為 21 歲人口數；單位人', '官方頁列逐歲欄名；精確年齡計算及月基準定義未取得'), '22歲數量': ('欄位標示為 22 歲人口數；單位人', '官方頁列逐歲欄名；精確年齡計算及月基準定義未取得'), '23歲數量': ('欄位標示為 23 歲人口數；單位人', '官方頁列逐歲欄名；精確年齡計算及月基準定義未取得'), '24歲數量': ('欄位標示為 24 歲人口數；單位人', '官方頁列逐歲欄名；精確年齡計算及月基準定義未取得'), '25歲數量': ('欄位標示為 25 歲人口數；單位人', '官方頁列逐歲欄名；精確年齡計算及月基準定義未取得'), '26歲數量': ('欄位標示為 26 歲人口數；單位人', '官方頁列逐歲欄名；精確年齡計算及月基準定義未取得'), '27歲數量': ('欄位標示為 27 歲人口數；單位人', '官方頁列逐歲欄名；精確年齡計算及月基準定義未取得'), '28歲數量': ('欄位標示為 28 歲人口數；單位人', '官方頁列逐歲欄名；精確年齡計算及月基準定義未取得'), '29歲數量': ('欄位標示為 29 歲人口數；單位人', '官方頁列逐歲欄名；精確年齡計算及月基準定義未取得'), '30歲數量': ('欄位標示為 30 歲人口數；單位人', '官方頁列逐歲欄名；精確年齡計算及月基準定義未取得'), '31歲數量': ('欄位標示為 31 歲人口數；單位人', '官方頁列逐歲欄名；精確年齡計算及月基準定義未取得'), '32歲數量': ('欄位標示為 32 歲人口數；單位人', '官方頁列逐歲欄名；精確年齡計算及月基準定義未取得'), '33歲數量': ('欄位標示為 33 歲人口數；單位人', '官方頁列逐歲欄名；精確年齡計算及月基準定義未取得'), '34歲數量': ('欄位標示為 34 歲人口數；單位人', '官方頁列逐歲欄名；精確年齡計算及月基準定義未取得'), '35歲數量': ('欄位標示為 35 歲人口數；單位人', '官方頁列逐歲欄名；精確年齡計算及月基準定義未取得'), '36歲數量': ('欄位標示為 36 歲人口數；單位人', '官方頁列逐歲欄名；精確年齡計算及月基準定義未取得'), '37歲數量': ('欄位標示為 37 歲人口數；單位人', '官方頁列逐歲欄名；精確年齡計算及月基準定義未取得'), '38歲數量': ('欄位標示為 38 歲人口數；單位人', '官方頁列逐歲欄名；精確年齡計算及月基準定義未取得'), '39歲數量': ('欄位標示為 39 歲人口數；單位人', '官方頁列逐歲欄名；精確年齡計算及月基準定義未取得'), '40歲數量': ('欄位標示為 40 歲人口數；單位人', '官方頁列逐歲欄名；精確年齡計算及月基準定義未取得'), '41歲數量': ('欄位標示為 41 歲人口數；單位人', '官方頁列逐歲欄名；精確年齡計算及月基準定義未取得'), '42歲數量': ('欄位標示為 42 歲人口數；單位人', '官方頁列逐歲欄名；精確年齡計算及月基準定義未取得'), '43歲數量': ('欄位標示為 43 歲人口數；單位人', '官方頁列逐歲欄名；精確年齡計算及月基準定義未取得'), '44歲數量': ('欄位標示為 44 歲人口數；單位人', '官方頁列逐歲欄名；精確年齡計算及月基準定義未取得'), '45歲數量': ('欄位標示為 45 歲人口數；單位人', '官方頁列逐歲欄名；精確年齡計算及月基準定義未取得'), '46歲數量': ('欄位標示為 46 歲人口數；單位人', '官方頁列逐歲欄名；精確年齡計算及月基準定義未取得'), '47歲數量': ('欄位標示為 47 歲人口數；單位人', '官方頁列逐歲欄名；精確年齡計算及月基準定義未取得'), '48歲數量': ('欄位標示為 48 歲人口數；單位人', '官方頁列逐歲欄名；精確年齡計算及月基準定義未取得'), '49歲數量': ('欄位標示為 49 歲人口數；單位人', '官方頁列逐歲欄名；精確年齡計算及月基準定義未取得'), '50歲數量': ('欄位標示為 50 歲人口數；單位人', '官方頁列逐歲欄名；精確年齡計算及月基準定義未取得'), '51歲數量': ('欄位標示為 51 歲人口數；單位人', '官方頁列逐歲欄名；精確年齡計算及月基準定義未取得'), '52歲數量': ('欄位標示為 52 歲人口數；單位人', '官方頁列逐歲欄名；精確年齡計算及月基準定義未取得'), '53歲數量': ('欄位標示為 53 歲人口數；單位人', '官方頁列逐歲欄名；精確年齡計算及月基準定義未取得'), '54歲數量': ('欄位標示為 54 歲人口數；單位人', '官方頁列逐歲欄名；精確年齡計算及月基準定義未取得'), '55歲數量': ('欄位標示為 55 歲人口數；單位人', '官方頁列逐歲欄名；精確年齡計算及月基準定義未取得'), '56歲數量': ('欄位標示為 56 歲人口數；單位人', '官方頁列逐歲欄名；精確年齡計算及月基準定義未取得'), '57歲數量': ('欄位標示為 57 歲人口數；單位人', '官方頁列逐歲欄名；精確年齡計算及月基準定義未取得'), '58歲數量': ('欄位標示為 58 歲人口數；單位人', '官方頁列逐歲欄名；精確年齡計算及月基準定義未取得'), '59歲數量': ('欄位標示為 59 歲人口數；單位人', '官方頁列逐歲欄名；精確年齡計算及月基準定義未取得'), '60歲數量': ('欄位標示為 60 歲人口數；單位人', '官方頁列逐歲欄名；精確年齡計算及月基準定義未取得'), '61歲數量': ('欄位標示為 61 歲人口數；單位人', '官方頁列逐歲欄名；精確年齡計算及月基準定義未取得'), '62歲數量': ('欄位標示為 62 歲人口數；單位人', '官方頁列逐歲欄名；精確年齡計算及月基準定義未取得'), '63歲數量': ('欄位標示為 63 歲人口數；單位人', '官方頁列逐歲欄名；精確年齡計算及月基準定義未取得'), '64歲數量': ('欄位標示為 64 歲人口數；單位人', '官方頁列逐歲欄名；精確年齡計算及月基準定義未取得'), '65歲數量': ('欄位標示為 65 歲人口數；單位人', '官方頁列逐歲欄名；精確年齡計算及月基準定義未取得'), '66歲數量': ('欄位標示為 66 歲人口數；單位人', '官方頁列逐歲欄名；精確年齡計算及月基準定義未取得'), '67歲數量': ('欄位標示為 67 歲人口數；單位人', '官方頁列逐歲欄名；精確年齡計算及月基準定義未取得'), '68歲數量': ('欄位標示為 68 歲人口數；單位人', '官方頁列逐歲欄名；精確年齡計算及月基準定義未取得'), '69歲數量': ('欄位標示為 69 歲人口數；單位人', '官方頁列逐歲欄名；精確年齡計算及月基準定義未取得'), '70歲數量': ('欄位標示為 70 歲人口數；單位人', '官方頁列逐歲欄名；精確年齡計算及月基準定義未取得'), '71歲數量': ('欄位標示為 71 歲人口數；單位人', '官方頁列逐歲欄名；精確年齡計算及月基準定義未取得'), '72歲數量': ('欄位標示為 72 歲人口數；單位人', '官方頁列逐歲欄名；精確年齡計算及月基準定義未取得'), '73歲數量': ('欄位標示為 73 歲人口數；單位人', '官方頁列逐歲欄名；精確年齡計算及月基準定義未取得'), '74歲數量': ('欄位標示為 74 歲人口數；單位人', '官方頁列逐歲欄名；精確年齡計算及月基準定義未取得'), '75歲數量': ('欄位標示為 75 歲人口數；單位人', '官方頁列逐歲欄名；精確年齡計算及月基準定義未取得'), '76歲數量': ('欄位標示為 76 歲人口數；單位人', '官方頁列逐歲欄名；精確年齡計算及月基準定義未取得'), '77歲數量': ('欄位標示為 77 歲人口數；單位人', '官方頁列逐歲欄名；精確年齡計算及月基準定義未取得'), '78歲數量': ('欄位標示為 78 歲人口數；單位人', '官方頁列逐歲欄名；精確年齡計算及月基準定義未取得'), '79歲數量': ('欄位標示為 79 歲人口數；單位人', '官方頁列逐歲欄名；精確年齡計算及月基準定義未取得'), '80歲數量': ('欄位標示為 80 歲人口數；單位人', '官方頁列逐歲欄名；精確年齡計算及月基準定義未取得'), '81歲數量': ('欄位標示為 81 歲人口數；單位人', '官方頁列逐歲欄名；精確年齡計算及月基準定義未取得'), '82歲數量': ('欄位標示為 82 歲人口數；單位人', '官方頁列逐歲欄名；精確年齡計算及月基準定義未取得'), '83歲數量': ('欄位標示為 83 歲人口數；單位人', '官方頁列逐歲欄名；精確年齡計算及月基準定義未取得'), '84歲數量': ('欄位標示為 84 歲人口數；單位人', '官方頁列逐歲欄名；精確年齡計算及月基準定義未取得'), '85歲數量': ('欄位標示為 85 歲人口數；單位人', '官方頁列逐歲欄名；精確年齡計算及月基準定義未取得'), '86歲數量': ('欄位標示為 86 歲人口數；單位人', '官方頁列逐歲欄名；精確年齡計算及月基準定義未取得'), '87歲數量': ('欄位標示為 87 歲人口數；單位人', '官方頁列逐歲欄名；精確年齡計算及月基準定義未取得'), '88歲數量': ('欄位標示為 88 歲人口數；單位人', '官方頁列逐歲欄名；精確年齡計算及月基準定義未取得'), '89歲數量': ('欄位標示為 89 歲人口數；單位人', '官方頁列逐歲欄名；精確年齡計算及月基準定義未取得'), '90歲數量': ('欄位標示為 90 歲人口數；單位人', '官方頁列逐歲欄名；精確年齡計算及月基準定義未取得'), '91歲數量': ('欄位標示為 91 歲人口數；單位人', '官方頁列逐歲欄名；精確年齡計算及月基準定義未取得'), '92歲數量': ('欄位標示為 92 歲人口數；單位人', '官方頁列逐歲欄名；精確年齡計算及月基準定義未取得'), '93歲數量': ('欄位標示為 93 歲人口數；單位人', '官方頁列逐歲欄名；精確年齡計算及月基準定義未取得'), '94歲數量': ('欄位標示為 94 歲人口數；單位人', '官方頁列逐歲欄名；精確年齡計算及月基準定義未取得'), '95歲數量': ('欄位標示為 95 歲人口數；單位人', '官方頁列逐歲欄名；精確年齡計算及月基準定義未取得'), '96歲數量': ('欄位標示為 96 歲人口數；單位人', '官方頁列逐歲欄名；精確年齡計算及月基準定義未取得'), '97歲數量': ('欄位標示為 97 歲人口數；單位人', '官方頁列逐歲欄名；精確年齡計算及月基準定義未取得'), '98歲數量': ('欄位標示為 98 歲人口數；單位人', '官方頁列逐歲欄名；精確年齡計算及月基準定義未取得'), '99歲數量': ('欄位標示為 99 歲人口數；單位人', '官方頁列逐歲欄名；精確年齡計算及月基準定義未取得'), '100歲以上': ('欄名所示 100 歲以上人口數，非單一 100 歲人口', '官方欄名；與 0–99 歲共 101 個年齡組；基準定義待確認')}
show_table('表 4：完整 107 欄定義、型態、缺失及相異值',schema(df,DEFINITIONS),block=18)
show_table('表 5：原始人口描述統計；非年度人口衍生表',df.describe(include='all').T,block=25)


**表 4：完整 107 欄定義、型態、缺失及相異值**

,欄位,讀入型態,定義或可觀察內容,定義依據及限制,層級,缺失數,非缺失相異值
0,年份,int64,原始年碼為民國年，112–115；研究使用 113、114,官方頁列年份；實際值及年月完整性見輸出,原始紀錄,0,4
1,月份,int64,人口統計月份；月內基準日／結算方式尚未取得,官方頁列月份，未提供本機版本的詳細基準說明,原始紀錄,0,12
2,區域代碼,string,字串保留原碼；本機市／區為 8 碼、里為 11 碼,本次依原碼長度、區域別及前綴的檔內診斷；正式代碼規格仍待官方文件,原始紀錄,0,469
3,區域別,str,區域名稱；市總計、行政區或行政里,原始值觀測；糖?里為原字串，不以換編碼修補,原始紀錄,0,467
4,性別,str,計、男、女；計為合計，不能再次與男／女相加,原始分類及逐欄男＋女＝計檢查,原始紀錄,0,3
5,總計,int64,該年／月／區域／性別的設籍人口合計,官方說明依戶籍登記產製；月內基準與本機版本尚待確認,原始紀錄,0,10458
6,0歲數量,int64,欄位標示為 0 歲人口數；單位人,官方頁列逐歲欄名；精確年齡計算及月基準定義未取得,原始紀錄,0,1110
7,1歲數量,int64,欄位標示為 1 歲人口數；單位人,官方頁列逐歲欄名；精確年齡計算及月基準定義未取得,原始紀錄,0,1127
8,2歲數量,int64,欄位標示為 2 歲人口數；單位人,官方頁列逐歲欄名；精確年齡計算及月基準定義未取得,原始紀錄,0,1144
9,3歲數量,int64,欄位標示為 3 歲人口數；單位人,官方頁列逐歲欄名；精確年齡計算及月基準定義未取得,原始紀錄,0,1195


,欄位,讀入型態,定義或可觀察內容,定義依據及限制,層級,缺失數,非缺失相異值
18,12歲數量,int64,欄位標示為 12 歲人口數；單位人,官方頁列逐歲欄名；精確年齡計算及月基準定義未取得,原始紀錄,0,1539
19,13歲數量,int64,欄位標示為 13 歲人口數；單位人,官方頁列逐歲欄名；精確年齡計算及月基準定義未取得,原始紀錄,0,1459
20,14歲數量,int64,欄位標示為 14 歲人口數；單位人,官方頁列逐歲欄名；精確年齡計算及月基準定義未取得,原始紀錄,0,1371
21,15歲數量,int64,欄位標示為 15 歲人口數；單位人,官方頁列逐歲欄名；精確年齡計算及月基準定義未取得,原始紀錄,0,1311
22,16歲數量,int64,欄位標示為 16 歲人口數；單位人,官方頁列逐歲欄名；精確年齡計算及月基準定義未取得,原始紀錄,0,1261
23,17歲數量,int64,欄位標示為 17 歲人口數；單位人,官方頁列逐歲欄名；精確年齡計算及月基準定義未取得,原始紀錄,0,1209
24,18歲數量,int64,欄位標示為 18 歲人口數；單位人,官方頁列逐歲欄名；精確年齡計算及月基準定義未取得,原始紀錄,0,1190
25,19歲數量,int64,欄位標示為 19 歲人口數；單位人,官方頁列逐歲欄名；精確年齡計算及月基準定義未取得,原始紀錄,0,1178
26,20歲數量,int64,欄位標示為 20 歲人口數；單位人,官方頁列逐歲欄名；精確年齡計算及月基準定義未取得,原始紀錄,0,1207
27,21歲數量,int64,欄位標示為 21 歲人口數；單位人,官方頁列逐歲欄名；精確年齡計算及月基準定義未取得,原始紀錄,0,1216


,欄位,讀入型態,定義或可觀察內容,定義依據及限制,層級,缺失數,非缺失相異值
36,30歲數量,int64,欄位標示為 30 歲人口數；單位人,官方頁列逐歲欄名；精確年齡計算及月基準定義未取得,原始紀錄,0,1307
37,31歲數量,int64,欄位標示為 31 歲人口數；單位人,官方頁列逐歲欄名；精確年齡計算及月基準定義未取得,原始紀錄,0,1297
38,32歲數量,int64,欄位標示為 32 歲人口數；單位人,官方頁列逐歲欄名；精確年齡計算及月基準定義未取得,原始紀錄,0,1317
39,33歲數量,int64,欄位標示為 33 歲人口數；單位人,官方頁列逐歲欄名；精確年齡計算及月基準定義未取得,原始紀錄,0,1377
40,34歲數量,int64,欄位標示為 34 歲人口數；單位人,官方頁列逐歲欄名；精確年齡計算及月基準定義未取得,原始紀錄,0,1414
41,35歲數量,int64,欄位標示為 35 歲人口數；單位人,官方頁列逐歲欄名；精確年齡計算及月基準定義未取得,原始紀錄,0,1464
42,36歲數量,int64,欄位標示為 36 歲人口數；單位人,官方頁列逐歲欄名；精確年齡計算及月基準定義未取得,原始紀錄,0,1441
43,37歲數量,int64,欄位標示為 37 歲人口數；單位人,官方頁列逐歲欄名；精確年齡計算及月基準定義未取得,原始紀錄,0,1458
44,38歲數量,int64,欄位標示為 38 歲人口數；單位人,官方頁列逐歲欄名；精確年齡計算及月基準定義未取得,原始紀錄,0,1510
45,39歲數量,int64,欄位標示為 39 歲人口數；單位人,官方頁列逐歲欄名；精確年齡計算及月基準定義未取得,原始紀錄,0,1552


,欄位,讀入型態,定義或可觀察內容,定義依據及限制,層級,缺失數,非缺失相異值
54,48歲數量,int64,欄位標示為 48 歲人口數；單位人,官方頁列逐歲欄名；精確年齡計算及月基準定義未取得,原始紀錄,0,1611
55,49歲數量,int64,欄位標示為 49 歲人口數；單位人,官方頁列逐歲欄名；精確年齡計算及月基準定義未取得,原始紀錄,0,1597
56,50歲數量,int64,欄位標示為 50 歲人口數；單位人,官方頁列逐歲欄名；精確年齡計算及月基準定義未取得,原始紀錄,0,1494
57,51歲數量,int64,欄位標示為 51 歲人口數；單位人,官方頁列逐歲欄名；精確年齡計算及月基準定義未取得,原始紀錄,0,1475
58,52歲數量,int64,欄位標示為 52 歲人口數；單位人,官方頁列逐歲欄名；精確年齡計算及月基準定義未取得,原始紀錄,0,1460
59,53歲數量,int64,欄位標示為 53 歲人口數；單位人,官方頁列逐歲欄名；精確年齡計算及月基準定義未取得,原始紀錄,0,1536
60,54歲數量,int64,欄位標示為 54 歲人口數；單位人,官方頁列逐歲欄名；精確年齡計算及月基準定義未取得,原始紀錄,0,1524
61,55歲數量,int64,欄位標示為 55 歲人口數；單位人,官方頁列逐歲欄名；精確年齡計算及月基準定義未取得,原始紀錄,0,1489
62,56歲數量,int64,欄位標示為 56 歲人口數；單位人,官方頁列逐歲欄名；精確年齡計算及月基準定義未取得,原始紀錄,0,1459
63,57歲數量,int64,欄位標示為 57 歲人口數；單位人,官方頁列逐歲欄名；精確年齡計算及月基準定義未取得,原始紀錄,0,1458


,欄位,讀入型態,定義或可觀察內容,定義依據及限制,層級,缺失數,非缺失相異值
72,66歲數量,int64,欄位標示為 66 歲人口數；單位人,官方頁列逐歲欄名；精確年齡計算及月基準定義未取得,原始紀錄,0,1528
73,67歲數量,int64,欄位標示為 67 歲人口數；單位人,官方頁列逐歲欄名；精確年齡計算及月基準定義未取得,原始紀錄,0,1525
74,68歲數量,int64,欄位標示為 68 歲人口數；單位人,官方頁列逐歲欄名；精確年齡計算及月基準定義未取得,原始紀錄,0,1512
75,69歲數量,int64,欄位標示為 69 歲人口數；單位人,官方頁列逐歲欄名；精確年齡計算及月基準定義未取得,原始紀錄,0,1531
76,70歲數量,int64,欄位標示為 70 歲人口數；單位人,官方頁列逐歲欄名；精確年齡計算及月基準定義未取得,原始紀錄,0,1479
77,71歲數量,int64,欄位標示為 71 歲人口數；單位人,官方頁列逐歲欄名；精確年齡計算及月基準定義未取得,原始紀錄,0,1450
78,72歲數量,int64,欄位標示為 72 歲人口數；單位人,官方頁列逐歲欄名；精確年齡計算及月基準定義未取得,原始紀錄,0,1445
79,73歲數量,int64,欄位標示為 73 歲人口數；單位人,官方頁列逐歲欄名；精確年齡計算及月基準定義未取得,原始紀錄,0,1439
80,74歲數量,int64,欄位標示為 74 歲人口數；單位人,官方頁列逐歲欄名；精確年齡計算及月基準定義未取得,原始紀錄,0,1459
81,75歲數量,int64,欄位標示為 75 歲人口數；單位人,官方頁列逐歲欄名；精確年齡計算及月基準定義未取得,原始紀錄,0,1430


... (3 outputs left out)

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
19歲數量,64722.0,NaN,NaN,NaN,80.251228,637.950824,0.0,16.0,24.0,37.0,19751.0
20歲數量,64722.0,NaN,NaN,NaN,81.65514,649.95399,0.0,17.0,25.0,37.0,21543.0
21歲數量,64722.0,NaN,NaN,NaN,84.451747,672.361234,0.0,17.0,26.0,38.0,21734.0
22歲數量,64722.0,NaN,NaN,NaN,90.483823,723.078654,0.0,18.0,27.0,41.0,26333.0
23歲數量,64722.0,NaN,NaN,NaN,97.349217,777.875521,0.0,20.0,30.0,44.0,26530.0
24歲數量,64722.0,NaN,NaN,NaN,100.884491,804.254383,0.0,21.0,31.0,46.0,26316.0
25歲數量,64722.0,NaN,NaN,NaN,106.005655,843.242147,0.0,22.0,32.0,48.0,27827.0
26歲數量,64722.0,NaN,NaN,NaN,109.322703,870.809334,0.0,23.0,33.0,49.0,28411.0
27歲數量,64722.0,NaN,NaN,NaN,113.568647,905.367,0.0,24.0,35.0,51.0,29287.0
28歲數量,64722.0,NaN,NaN,NaN,119.488458,950.055525,0.0,25.0,36.0,54.0,29739.0


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
44歲數量,64722.0,NaN,NaN,NaN,180.901919,1437.572586,2.0,38.0,55.0,81.0,43851.0
45歲數量,64722.0,NaN,NaN,NaN,176.739409,1404.976823,1.0,37.0,54.0,80.0,43058.0
46歲數量,64722.0,NaN,NaN,NaN,175.891443,1398.565909,0.0,37.0,53.0,80.0,45097.0
47歲數量,64722.0,NaN,NaN,NaN,172.077593,1369.343129,0.0,36.0,52.0,78.0,45058.0
48歲數量,64722.0,NaN,NaN,NaN,165.616576,1318.39786,0.0,35.0,50.0,75.0,44276.0
49歲數量,64722.0,NaN,NaN,NaN,162.444702,1292.818765,1.0,34.0,49.0,74.0,42962.0
50歲數量,64722.0,NaN,NaN,NaN,154.577732,1228.586455,1.0,33.0,47.0,70.0,40176.0
51歲數量,64722.0,NaN,NaN,NaN,152.134606,1209.027183,1.0,32.0,46.0,69.0,36974.0
52歲數量,64722.0,NaN,NaN,NaN,153.136461,1217.659651,1.0,32.0,46.0,70.0,38092.0
53歲數量,64722.0,NaN,NaN,NaN,155.011495,1233.159894,2.0,33.0,47.0,70.0,38182.0


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
69歲數量,64722.0,NaN,NaN,NaN,152.624733,1214.342514,1.0,32.0,47.0,69.0,37483.0
70歲數量,64722.0,NaN,NaN,NaN,149.238621,1187.192268,0.0,32.0,46.0,68.0,36978.0
71歲數量,64722.0,NaN,NaN,NaN,145.202929,1154.97063,0.0,31.0,45.0,66.0,35645.0
72歲數量,64722.0,NaN,NaN,NaN,140.265597,1116.342886,0.0,30.0,43.0,64.0,33717.0
73歲數量,64722.0,NaN,NaN,NaN,132.996663,1060.501689,0.0,28.0,41.0,61.0,33163.0
74歲數量,64722.0,NaN,NaN,NaN,123.812923,989.37289,0.0,26.0,38.0,57.0,32334.0
75歲數量,64722.0,NaN,NaN,NaN,110.831742,887.965425,0.0,23.0,34.0,51.0,31260.0
76歲數量,64722.0,NaN,NaN,NaN,97.364049,782.063582,0.0,20.0,30.0,45.0,26689.0
77歲數量,64722.0,NaN,NaN,NaN,81.98285,665.945777,0.0,16.0,25.0,38.0,25228.0
78歲數量,64722.0,NaN,NaN,NaN,69.989988,566.250707,0.0,14.0,21.0,32.0,20833.0


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
94歲數量,64722.0,NaN,NaN,NaN,10.376379,82.989895,0.0,2.0,3.0,5.0,2548.0
95歲數量,64722.0,NaN,NaN,NaN,8.137851,65.044924,0.0,1.0,2.0,4.0,2003.0
96歲數量,64722.0,NaN,NaN,NaN,6.311115,50.48155,0.0,1.0,2.0,3.0,1545.0
97歲數量,64722.0,NaN,NaN,NaN,4.673589,37.420631,0.0,0.0,1.0,2.0,1183.0
98歲數量,64722.0,NaN,NaN,NaN,3.298044,26.449492,0.0,0.0,1.0,2.0,862.0
99歲數量,64722.0,NaN,NaN,NaN,2.315936,18.609457,0.0,0.0,1.0,1.0,613.0
100歲以上,64722.0,NaN,NaN,NaN,4.790396,38.327312,0.0,0.0,1.0,2.0,1229.0


#### 月份、層級、唯一性與人口加總一致性


In [6]:
AGE_COLUMNS=[f'{i}歲數量' for i in range(100)]+['100歲以上']
NUMERIC=['總計']+AGE_COLUMNS
KEY=['年份','月份','區域代碼','性別']
code_length=df['區域代碼'].str.len()
city=df.loc[df['區域代碼'].eq('63000000')]
district=df.loc[code_length.eq(8)&df['區域別'].ne('總計')]
village=df.loc[code_length.eq(11)]
print({'city_codes':city['區域代碼'].nunique(),'district_codes':district['區域代碼'].nunique(),
       'village_codes':village['區域代碼'].nunique(),'other_code_lengths':int((~code_length.isin([8,11])).sum()),
       'key_duplicates':int(df.duplicated(KEY).sum()),'full_row_duplicates':int(df.duplicated().sum()),
       'key_missing_cells':int(df[KEY].isna().to_numpy().sum()),'all_missing_cells':int(df.isna().to_numpy().sum()),
       'age_column_count':len(AGE_COLUMNS),'age_sum_mismatch_rows':int(df[AGE_COLUMNS].sum(axis=1).ne(df['總計']).sum()),
       'numeric_negative_cells':int(df[NUMERIC].lt(0).to_numpy().sum()),
       'numeric_missing_cells':int(df[NUMERIC].isna().to_numpy().sum())})
show_table('表 6：市／區辨識的原始碼與名稱；層級推斷尚非正式代碼 crosswalk',
           df.loc[code_length.eq(8),['區域代碼','區域別']].drop_duplicates())
coverage=df.groupby(['年份','月份']).agg(原始列數=('區域代碼','size'),區域數=('區域代碼','nunique'))
show_table('表 7：原始檔所有年月涵蓋',coverage,block=24)
research=village.loc[village['年份'].isin([113,114])]
coverage24=research.groupby(['年份','月份']).agg(原始里層級列數=('區域代碼','size'),里數=('區域代碼','nunique'))
coverage24['計的里數']=research.loc[research['性別'].eq('計')].groupby(['年份','月份'])['區域代碼'].nunique()
show_table('表 8：研究 24 個月完整性；每月須 456 里×3 性別',coverage24)
print({'research_year_month_pairs':len(coverage24),'all_months_456_villages':bool(coverage24['里數'].eq(456).all()),
       'all_months_1368_rows':bool(coverage24['原始里層級列數'].eq(1368).all()),
       'all_months_456_total_records':bool(coverage24['計的里數'].eq(456).all())})
wide=df.pivot(index=['年份','月份','區域代碼'],columns='性別',values=NUMERIC)
sex_difference=wide.xs('男',level='性別',axis=1)+wide.xs('女',level='性別',axis=1)-wide.xs('計',level='性別',axis=1)
print({'male_plus_female_equals_total_mismatch_cells':int(sex_difference.ne(0).to_numpy().sum()),
       'missing_gender_cells':int(wide.isna().to_numpy().sum()),
       'denominator':'all original year-month-region groups × total/101 age columns'})
sets={year:set(research.loc[research['年份'].eq(year),'區域代碼']) for year in [113,114]}
print({'2024_village_code_count':len(sets[113]),'2025_village_code_count':len(sets[114]),
       'code_sets_identical':sets[113]==sets[114]})


{'city_codes': 1, 'district_codes': 12, 'village_codes': 456, 'other_code_lengths': 0, 'key_duplicates': 0, 'full_row_duplicates': 0, 'key_missing_cells': 0, 'all_missing_cells': 0, 'age_column_count': 101, 'age_sum_mismatch_rows': 0, 'numeric_negative_cells': 0, 'numeric_missing_cells': 0}


**表 6：市／區辨識的原始碼與名稱；層級推斷尚非正式代碼 crosswalk**

,區域代碼,區域別
0,63000000,總計
3,63000010,松山區
105,63000020,信義區
231,63000030,大安區
393,63000040,中山區
522,63000050,中正區
618,63000060,大同區
696,63000070,萬華區
807,63000080,文山區
939,63000090,南港區


**表 7：原始檔所有年月涵蓋**

原始列數  區域數
年份  月份           
112 1   1407  469
    2   1407  469
    3   1407  469
    4   1407  469
    5   1407  469
    6   1407  469
    7   1407  469
    8   1407  469
    9   1407  469
    10  1407  469
    11  1407  469
    12  1407  469
113 1   1407  469
    2   1407  469
    3   1407  469
    4   1407  469
    5   1407  469
    6   1407  469
    7   1407  469
    8   1407  469
    9   1407  469
    10  1407  469
    11  1407  469
    12  1407  469

原始列數  區域數
年份  月份           
114 1   1407  469
    2   1407  469
    3   1407  469
    4   1407  469
    5   1407  469
    6   1407  469
    7   1407  469
    8   1407  469
    9   1407  469
    10  1407  469
    11  1407  469
    12  1407  469
115 1   1407  469
    2   1407  469
    3   1407  469
    4   1407  469
    5   1407  469
    6   1407  469
    7   1407  469
    8   1407  469
    9   1407  469
    10  1407  469

**表 8：研究 24 個月完整性；每月須 456 里×3 性別**

原始里層級列數   里數  計的里數
年份  月份                    
113 1      1368  456   456
    2      1368  456   456
    3      1368  456   456
    4      1368  456   456
    5      1368  456   456
    6      1368  456   456
    7      1368  456   456
    8      1368  456   456
    9      1368  456   456
    10     1368  456   456
    11     1368  456   456
    12     1368  456   456
114 1      1368  456   456
    2      1368  456   456
    3      1368  456   456
    4      1368  456   456
    5      1368  456   456
    6      1368  456   456
    7      1368  456   456
    8      1368  456   456
    9      1368  456   456
    10     1368  456   456
    11     1368  456   456
    12     1368  456   456

{'research_year_month_pairs': 24, 'all_months_456_villages': True, 'all_months_1368_rows': True, 'all_months_456_total_records': True}


{'male_plus_female_equals_total_mismatch_cells': 0, 'missing_gender_cells': 0, 'denominator': 'all original year-month-region groups × total/101 age columns'}
{'2024_village_code_count': 456, '2025_village_code_count': 456, 'code_sets_identical': True}


#### 與里界的唯讀名稱診斷；不是正式 crosswalk


In [7]:
import geopandas as gpd
boundary=gpd.read_file(project_file('data/raw/G97_A_CAVLGE_P.shp'),engine='pyogrio',encoding='utf-8')
# In-memory hierarchy diagnostic only; no crosswalk is created or exported.
district_names=district[['區域代碼','區域別']].drop_duplicates().set_index('區域代碼')['區域別'].to_dict()
names=village[['區域代碼','區域別']].drop_duplicates()
observed_pairs={(district_names.get(code[:8]),name) for code,name in names.itertuples(index=False,name=None)}
boundary_pairs=set(boundary[['TNAME','VNAME']].itertuples(index=False,name=None))
print({'population_unique_district_name_pairs':len(observed_pairs),'boundary_name_pairs':len(boundary_pairs),
       'exact_name_matches':len(observed_pairs&boundary_pairs),
       'population_unmatched':sorted(observed_pairs-boundary_pairs),
       'boundary_unmatched':sorted(boundary_pairs-observed_pairs),
       'population_village_code_length':11,'boundary_CPTVID_length':10,
       'one_to_one_for_exact_pairs':len(observed_pairs&boundary_pairs)==455})
show_table('表 9：含糖字的原始人口紀錄名稱與代碼（未更名）',
           names.loc[names['區域別'].str.contains('糖',na=False)])
show_table('表 10：含糖字的原始里界名稱與代碼（未更名）',
           boundary.loc[boundary['VNAME'].str.contains('糖',na=False),['TNAME','VNAME','CPTVID']])
print({'literal_question_mark_in_raw_name':bool(names['區域別'].str.contains(r'\?',regex=True,na=False).any()),
       'formal_official_name':'糖廍里（沿用 Unit 0.2 歷史紀錄；本次未取得官方碼表列）',
       'official_code_crosswalk':'pending; do not truncate/pad codes or overwrite raw names'})


{'population_unique_district_name_pairs': 456, 'boundary_name_pairs': 456, 'exact_name_matches': 455, 'population_unmatched': [('萬華區', '糖?里')], 'boundary_unmatched': [('萬華區', '糖蔀里')], 'population_village_code_length': 11, 'boundary_CPTVID_length': 10, 'one_to_one_for_exact_pairs': True}


**表 9：含糖字的原始人口紀錄名稱與代碼（未更名）**

,區域代碼,區域別
741,63000070015,糖?里


**表 10：含糖字的原始里界名稱與代碼（未更名）**

,TNAME,VNAME,CPTVID
449,萬華區,糖蔀里,6300700015


{'literal_question_mark_in_raw_name': True, 'formal_official_name': '糖廍里（沿用 Unit 0.2 歷史紀錄；本次未取得官方碼表列）', 'official_code_crosswalk': 'pending; do not truncate/pad codes or overwrite raw names'}


#### 與既有診斷及研究用途的關係
本次重算支持 Unit 0.2 的 24 個月完整、里代碼集合一致及 455／456 精確名稱配對。
新增確認逐歲人口合計＝總計、男＋女＝計，檔內無負值或缺失；這些僅是算術與結構診斷，
不認證戶籍人口等同實住人口、交通活動或通勤車流，也不解決官方里代碼對照。
研究已確認里層級「計」的 12 個月平均作年度存量；年齡比例用全年各月年齡組人數合計除以總人口合計。
本次沒有計算或輸出年度人口衍生資料，年齡分組界線仍待後續確認。


## Cross-Dataset Documentation — 僅規劃、未正式整合

| 原始資料 | 原始觀察單位／時間／空間粒度 | 預計連接與可支持變數 | 整合前條件 |
| --- | --- | --- | --- |
| 2024、2025 事故 | 2024 為當事人紀錄；2025 僅第一當事人；事故時分與點位 | 第一當事人候選事件；驗證 CRS 後以點在行政里面歸里；A 事故組成、C 負擔 | 官方事件口徑、照明欄位跨年映射、缺失／異常及座標基準；邊界點／未匹配規則另審查。 |
| 人口 | 年×月×區域×性別，含市／區／里 | 里層級「計」；12 月平均存量、人口結構 B1 | 代碼 crosswalk、糖廍里例外、月資料基準日與里界適用性；不視為交通暴露人口。 |
| 行政里界 | 456 個里面單元、單一未認證年代快照 | CPTVID 候選空間 ID；人口名稱及代碼驗證後連接，並作事故歸里 | 適用年度、幾何重疊／縫隙、原 AREA 定義與重算口徑；不因里數相同認證兩年幾何一致。 |
| 都市計畫 | 細計面與點、未認證年代快照、未知 CRS | 確認後以面圖層分類與疊合支援 B2 分區比例；點圖層用途待確認 | 原包、CRS、版本、143 筆無效幾何、重複及部分重疊、非一般分區、正式分類表。 |

| 規劃資料集 | 來源及變數 | 狀態與解釋限制 |
| --- | --- | --- |
| Dataset A | 事故；早／晚尖峰、深夜、週末、路口、追撞、側撞、人車、多人受傷組成 | 尚未建立；保留分子／有效分母／缺失／涵蓋率，不用總人口作事故組成分母。 |
| Dataset B | 人口 B1＋通過驗證的都市計畫 B2；其他道路環境 B3 尚未取得或選用 | 尚未建立；不納入事故結果，不以事故中的道路情境代表全里環境；分區不是實際土地使用。 |
| Dataset C | 事故數、傷亡、每件受傷數、事故空間密度、低樣本標記 | 尚未建立；24 小時與 2–30 日死亡分列，缺值不補成 0；每平方公里事故數不是暴露校正事故率。 |

village-year 為後續基礎資料結構；兩年比例合併須合計分子／有效分母再重算，
不能直接平均年度比例。零事故里及低樣本門檻仍待審查。本文件未建立 crosswalk、
年度人口表或 A／B／C，亦未執行事故歸里、土地使用疊合、分群或統計推論。

其他原始資料說明：
[事故](taipei_2024_2025_crashes_raw_documentation.ipynb)、[人口](taipei_population_age_raw_documentation.ipynb)、[里界](taipei_village_boundaries_raw_documentation.ipynb)、[都市計畫](taipei_urban_planning_raw_documentation.ipynb).


In [8]:
finish_check()


{'input_hashes_unchanged': True, 'input_files_checked': 6, 'interpreter_matches_project_venv': True, 'python_version': '3.12.2', 'scope': 'read-only raw documentation; no processed dataset or model output'}
